# Cómputo del Modelo Global — Aprendizaje Federado

Este notebook agrega los pesos de los 3 modelos locales en un modelo global.

**Prerequisito:** coloca los archivos `weights_0.npz`, `weights_1.npz` y `weights_2.npz`
(generados por cada miembro en `training_local.ipynb`) en este mismo directorio.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from TheModel import build_model

# Conjunto de prueba global (mismo para todos)
_, (x_test_raw, y_test) = tf.keras.datasets.mnist.load_data()
x_test = np.expand_dims(x_test_raw / 255.0, -1)

# Número de muestras de cada miembro (para ponderar FedAvg)
n_samples = [20000, 20000, 20000]

def load_weights(filepath):
    """Carga pesos de un .npz y los devuelve como lista ordenada de arrays."""
    data = np.load(filepath)
    return [data[k] for k in sorted(data.files)]

# Carga los pesos de los 3 miembros
all_weights = [load_weights(f'weights_{i}.npz') for i in range(3)]
n_layers = len(all_weights[0])
print(f'Modelos cargados: {len(all_weights)} | Capas con pesos: {n_layers}')

In [ ]:
def evaluate(weights, name):
    """Asigna pesos a un modelo limpio, evalúa en test y muestra el reporte."""
    model = build_model()
    model.set_weights(weights)
    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    print(f'\n════ {name} ════')
    print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))
    return np.mean(y_pred == y_test)

## 1. FedAvg — Promedio Ponderado de Pesos

El método más simple: cada capa del modelo global es el promedio ponderado
de los pesos de cada miembro, con pesos proporcionales al tamaño de su dataset.

In [ ]:
total = sum(n_samples)

# Para cada capa: suma ponderada de pesos / total de muestras
fedavg_weights = [
    sum(n * w[layer] for n, w in zip(n_samples, all_weights)) / total
    for layer in range(n_layers)
]

acc_fedavg = evaluate(fedavg_weights, 'FedAvg')

## 2. FedMedian — Mediana Coordenada a Coordenada

En lugar de promediar, toma la mediana elemento a elemento de los pesos.
Es más robusto ante clientes con datos ruidosos o comportamiento anómalo.

In [ ]:
# np.stack apila los arrays de cada miembro; np.median toma la mediana por posición
fedmedian_weights = [
    np.median(np.stack([w[layer] for w in all_weights]), axis=0)
    for layer in range(n_layers)
]

acc_fedmedian = evaluate(fedmedian_weights, 'FedMedian')

## 3. FedAcc — Promedio Ponderado por Precisión Local

Cada modelo local se pondera según su precisión en el conjunto de prueba global.
Los modelos que aprendieron mejor contribuyen más al modelo global.
Esto mejora FedAvg cuando los clientes tienen calidades de entrenamiento distintas.

In [ ]:
# Calcula la precisión de cada modelo local en el test set
local_accs = []
for i, weights in enumerate(all_weights):
    model = build_model()
    model.set_weights(weights)
    y_pred_i = np.argmax(model.predict(x_test, verbose=0), axis=1)
    acc_i = np.mean(y_pred_i == y_test)
    local_accs.append(acc_i)
    print(f'Precisión local — Miembro {i}: {acc_i:.4f}')

# Pondera cada modelo por su precisión
total_acc = sum(local_accs)
fedacc_weights = [
    sum(a * w[layer] for a, w in zip(local_accs, all_weights)) / total_acc
    for layer in range(n_layers)
]

acc_fedacc = evaluate(fedacc_weights, 'FedAcc (ponderado por precisión)')

In [ ]:
# ─── Comparación de métodos ────────────────────────────────────────────
methods    = ['FedAvg', 'FedMedian', 'FedAcc']
accuracies = [acc_fedavg, acc_fedmedian, acc_fedacc]

plt.figure(figsize=(7, 4))
bars = plt.bar(methods, accuracies, color=['steelblue', 'seagreen', 'coral'])
plt.ylim(min(accuracies) - 0.01, 1.0)
plt.ylabel('Accuracy (test set)')
plt.title('Comparación de métodos de agregación federada')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
             f'{acc:.4f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()